# GeoSPARQL 1.0 Tutorial (rdflib-geosparql)
This tutorial showcases the functions of the GeoSPARQL 1.0 standard which have been implemented in rdflib-geosparql.

## Installing requirements

In [329]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet --break-system-packages
!jupyter labextension install jupyter-leaflet

import rdflib
import shapely.geometry
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
from ipyleaflet import *

mapcenter=(24, 115)
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in resultlist:
        for item in row:
            if isinstance(row[item],Literal) and row[item].datatype!=None:
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip()+"</a></td></tr>"
            elif isinstance(row[item],URIRef):
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
            else:
                res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip()+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows,lmap):
    layers=[]
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in rows:
        if row in resultlist:
            lmap.add(WKTLayer(wkt_string=resultlist[row],hover_style={"fillColor": "red"}))
    return lmap    

def processQueryResults(qres,rows=[],lmap=None):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if lmap!=None:
        getMapFromWKTResult(qres,rows,lmap)
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:54: UserWarning: An error occurred.
  warnings.warn("An error occurred.")
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:55: UserWarning: PermissionError: [Errno 13] Permission denied: '/usr/local/share/jupyter/lab/extensions/jupyter-leaflet-0.20.0.tgz'
  warnings.warn(msg[-1].strip())
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:56: UserWarning: See the log file for details: /tmp/jupyterlab-debug-rr773re3.log


## GeoSPARQL 1.0 Egenhofer Relations

### geof:ehContains Function

In [330]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?fLiteral (xsd:boolean(?ehContains) as ?AcontainsF)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:ehContains(?aLiteral, ?fLiteral) as ?ehContains)
}
"""),["aLiteral","fLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
GString: Point(-83.4 34.4)


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
fLiteral,Point(-83.4 34.4)
AcontainsF,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:ehCovers Function

In [331]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehCovers) as ?AcoversB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:ehCovers(?aLiteral, ?bLiteral) as ?ehCovers)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AcoversB,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:ehCoveredBy Function

In [332]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehCoveredBy) as ?BcoveredByA)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:ehCoveredBy(?bLiteral, ?aLiteral) as ?ehCoveredBy)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
BcoveredByA,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:ehDisjoint Function

In [333]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehDisjoint) as ?BdisjointB) (xsd:boolean(?ehDisjoint2) as ?BdisjointC)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:ehDisjoint(?bLiteral, ?bLiteral) as ?ehDisjoint)
  BIND (geof:ehDisjoint(?bLiteral, ?cLiteral) as ?ehDisjoint2)
}
"""),["bLiteral","cLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
POLYGON ((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
POLYGON ((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
GString: Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
POLYGON ((-83.6 34.1, -83.4 34.1, -83.

Variable,Value
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
BdisjointB,false
BdisjointC,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:ehEquals Function

In [334]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehEquals) as ?AequalsA) (xsd:boolean(?ehEquals2) as ?AequalsB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:ehEquals(?aLiteral, ?aLiteral) as ?ehEquals)
  BIND (geof:ehEquals(?aLiteral, ?bLiteral) as ?ehEquals2)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AequalsA,true
AequalsB,false


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:ehInside Function

In [335]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?insidee) as ?FinsideA)
WHERE {
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:ehInside(?fLiteral, ?aLiteral) as ?insidee)
}
"""),["aLiteral","fLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
GString: Point(-83.4 34.4)
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
True


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
FinsideA,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:ehMeet Function

In [336]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cLiteral (xsd:boolean(?ehMeet) as ?AmeetC)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:ehMeet(?aLiteral, ?cLiteral) as ?ehMeet)
}
"""),["aLiteral","cLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
GString: Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
AmeetC,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:ehOverlap Function

In [337]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?ehOverlap) as ?AoverlapD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:ehOverlap(?aLiteral, ?dLiteral) as ?ehOverlap)
}
"""),["aLiteral","dLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
AoverlapD,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

## GeoSPARQL 1.0 RCC8 Relations

### geof:rcc8dc Function

In [338]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?bLiteral ?cLiteral (xsd:boolean(?rcc8dc) as ?BdisjointB) (xsd:boolean(?rcc8dc2) as ?BdisjointC)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:rcc8dc(?bLiteral, ?bLiteral) as ?rcc8dc)
  BIND (geof:rcc8dc(?bLiteral, ?cLiteral) as ?rcc8dc2)
}
"""),["bLiteral","cLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
POLYGON ((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
POLYGON ((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
GString: Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
POLYGON ((-83.6 34.1, -83.4 34.1, -83.

Variable,Value
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
BdisjointB,false
BdisjointC,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:rcc8ec Function

In [339]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cLiteral (xsd:boolean(?sfTouches) as ?AtouchesC)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:rcc8ec(?aLiteral, ?cLiteral) as ?sfTouches)
}
"""),["aLiteral","cLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
GString: Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
AtouchesC,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:rcc8eq Function

In [340]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?rcc8eq) as ?AequalsA) (xsd:boolean(?rcc8eq2) as ?AequalsB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:rcc8eq(?aLiteral, ?aLiteral) as ?rcc8eq)
  BIND (geof:rcc8eq(?aLiteral, ?bLiteral) as ?rcc8eq2)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AequalsA,true
AequalsB,false


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:rcc8ntpp Function

In [341]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gLiteral (xsd:boolean(?rcc8ntppr) as ?Grcc8ntppA) (xsd:boolean(?rcc8ntppr2) as ?Arcc8ntppG)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:G geo:hasDefaultGeometry ?gGeom .
  ?gGeom geo:asWKT ?gLiteral .
  BIND (geof:rcc8ntpp(?gLiteral, ?aLiteral) as ?rcc8ntppr)
  BIND (geof:rcc8ntpp(?aLiteral, ?gLiteral) as ?rcc8ntppr2)
}
"""),["aLiteral","gLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
GString: Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
True
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
GString: Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
False


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gLiteral,"Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))"
Grcc8ntppA,true
Arcc8ntppG,false


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:rcc8ntppi Function

In [342]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gLiteral (xsd:boolean(?rcc8ntppir) as ?Arcc8ntppiG) (xsd:boolean(?rcc8ntppir2) as ?Grcc8ntppiA)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:G geo:hasDefaultGeometry ?gGeom .
  ?gGeom geo:asWKT ?gLiteral .
  BIND (geof:rcc8ntppi(?aLiteral, ?gLiteral) as ?rcc8ntppir)
  BIND (geof:rcc8ntppi(?gLiteral, ?aLiteral) as ?rcc8ntppir2)
}
"""),["aLiteral","gLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
GString: Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
GString: Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gLiteral,"Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))"
Arcc8ntppiG,true
Grcc8ntppiA,false


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:rcc8po Function

In [343]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?rcc8por) as ?Arcc8poD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:rcc8po(?aLiteral, ?dLiteral) as ?rcc8por)
}
"""),["aLiteral","dLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
Arcc8poD,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:rcc8tpp Function

In [344]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?rcc8tppr) as ?Arcc8tppB)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?aGeom geo:asWKT ?bLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:rcc8tpp(?aLiteral, ?bLiteral) as ?rcc8tppr)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
Arcc8tppB,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:rcc8tppi Function

In [345]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?rcc8tppir) as ?Arcc8tppiB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:rcc8tppi(?aLiteral, ?bLiteral) as ?rcc8tppir)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
Arcc8tppiB,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

## GeoSPARQL 1.0 Simple Feature Relations

### geof:sfContains Function

In [346]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?fLiteral (xsd:boolean(?sfContains) as ?AcontainsF)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:sfContains(?aLiteral, ?fLiteral) as ?sfContains)
}
"""),["aLiteral","fLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
GString: Point(-83.4 34.4)


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
fLiteral,Point(-83.4 34.4)
AcontainsF,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:sfCrosses Function

In [347]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?eLiteral (xsd:boolean(?sfCrosses) as ?EcrossesA)
WHERE {
  my:E geo:hasDefaultGeometry ?eGeom .
  ?eGeom geo:asWKT ?eLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:sfCrosses(?eLiteral, ?aLiteral) as ?sfCrosses)
}
"""),["aLiteral","eLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)
GString: LineString(-83.4 34.0, -83.3 34.3)
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
eLiteral,"LineString(-83.4 34.0, -83.3 34.3)"
EcrossesA,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:sfDisjoint Function

In [348]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral (xsd:boolean(?sfDisjoint) as ?CdisjointF)
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:sfDisjoint(?cLiteral, ?fLiteral) as ?sfDisjoint)
}
"""),["cLiteral","fLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
GString: Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
GString: Point(-83.4 34.4)
POLYGON ((-83.2 34.3, -83 34.3, -83 34.5, -83.2 34.5, -83.2 34.3))
POINT (-83.4 34.4)


Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
CdisjointF,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:sfEquals Function

In [349]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?sfEquals) as ?AequalsA) (xsd:boolean(?sfEquals2) as ?AequalsB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:sfEquals(?aLiteral, ?aLiteral) as ?sfEquals)
  BIND (geof:sfEquals(?aLiteral, ?bLiteral) as ?sfEquals2)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AequalsA,true
AequalsB,false


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:sfIntersects Function

In [350]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?sfIntersects) as ?AintersectsD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:sfIntersects(?aLiteral, ?dLiteral) as ?sfIntersects)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
AintersectsD,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:sfOverlaps Function

In [351]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?sfOverlaps) as ?AoverlapsD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:sfOverlaps(?aLiteral, ?dLiteral) as ?sfOverlaps)
}
"""),["aLiteral","dLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
AoverlapsD,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:sfTouches Function

In [352]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cLiteral (xsd:boolean(?sfTouches) as ?AtouchesC)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:sfTouches(?aLiteral, ?cLiteral) as ?sfTouches)
}
"""),["aLiteral","cLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
GString: Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
AtouchesC,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:sfWithin Function

In [353]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?sfWithin) as ?BwithinC)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:sfWithin(?bLiteral, ?aLiteral) as ?sfWithin)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
BwithinC,true


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

## GeoSPARQL 1.0 Non-topological query functions

### geof:buffer Function

In [354]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?buffer
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:buffer(?aLiteral,"5.0"^^xsd:double,<http://www.ontology-of-units-of-measure.org/resource/om-2/meter>) as ?buffer)
}
"""),["aLiteral","buffer"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GEOMTOLIT: POLYGON ((-88.6 34.1, -88.6 34.5, -88.50392640201615 35.47545161008064, -88.21939766255643 36.41341716182545, -87.75734806151272 37.277851165098014, -87.13553390593273 38.03553390593274, -86.37785116509801 38.65734806151273, -85.51341716182544 39.11939766255644, -84.57545161008063 39.403926402016154, -83.6 39.5, -83.2 39.5, -82.22454838991936 39.403926402016154, -81.28658283817455 39.11939766255644, -80.42214883490199 38.65734806151273, -79.66446609406727 38.03553390593274, -79.04265193848728 37.277851165098014, -78.58060233744357 36.41341716182545, -78.29607359798385 35.47545161008064, -78.2 34.5, -78.2 34.1, -78.29607359798385 33.12454838991936, -78.58060233744357 32.18658283817455, -79.04265193848728 31.32214883490199, -79.66446609406727 30.564466094067264, -80.422148834

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
buffer,"POLYGON ((-88.6 34.1, -88.6 34.5, -88.50392640201615 35.47545161008064, -88.21939766255643 36.41341716182545, -87.75734806151272 37.277851165098014, -87.13553390593273 38.03553390593274, -86.37785116509801 38.65734806151273, -85.51341716182544 39.11939766255644, -84.57545161008063 39.403926402016154, -83.6 39.5, -83.2 39.5, -82.22454838991936 39.403926402016154, -81.28658283817455 39.11939766255644, -80.42214883490199 38.65734806151273, -79.66446609406727 38.03553390593274, -79.04265193848728 37.277851165098014, -78.58060233744357 36.41341716182545, -78.29607359798385 35.47545161008064, -78.2 34.5, -78.2 34.1, -78.29607359798385 33.12454838991936, -78.58060233744357 32.18658283817455, -79.04265193848728 31.32214883490199, -79.66446609406727 30.564466094067264, -80.42214883490199 29.942651938487273, -81.28658283817455 29.480602337443568, -82.22454838991936 29.196073597983847, -83.2 29.1, -83.6 29.1, -84.57545161008063 29.196073597983847, -85.51341716182544 29.480602337443568, -86.37785116509801 29.942651938487273, -87.13553390593273 30.564466094067264, -87.75734806151272 31.32214883490199, -88.21939766255643 32.18658283817455, -88.50392640201615 33.12454838991936, -88.6 34.1))"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:boundary Function

In [355]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?boundary
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundary(?aLiteral) as ?boundary)
}
"""),["aLiteral","boundary"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GEOMTOLIT: LINESTRING (-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1)
<class 'shapely.geometry.linestring.LineString'>


Variable,Value
boundary,"LINESTRING (-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1)"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:convexHull Function

In [356]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?chull
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:convexHull(?aLiteral) as ?chull)
}
"""),["aLiteral","chull"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GEOMTOLIT: POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
chull,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:distance Function

In [357]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?distance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:distance(?cLiteral, ?fLiteral,"http://www.opengis.net/def/uom/OGC/1.0/meter"^^xsd:anyURI) as ?distance)
}
"""),["cLiteral","fLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
GString: Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
GString: Point(-83.4 34.4)


Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
distance,0.20000000000000284


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:difference Function

In [358]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?difference
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:difference(?aLiteral, ?dLiteral) as ?difference)
}
"""),["aLiteral","dLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
[(<POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))>, 'http://www.opengis.net/def/crs/OGC/1.3/CRS84'), (<POLYGON ((-83.3 34, -83.1 34, -83.1 34.2, -83.3 34.2, -83.3 34))>, 'http://www.opengis.net/def/crs/OGC/1.3/CRS84')]
POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1))
GEOMTOLIT: POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
difference,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1))"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:envelope Function

In [359]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?envelope
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:envelope(?aLiteral) as ?envelope)
}
"""),["aLiteral","envelope"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GEOMTOLIT: POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
envelope,"POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:geometryType Function

In [360]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?gtype
WHERE {
  my:A geo:hasDefaultGeometry ?geom .
  ?geom geo:asWKT ?literal .
  BIND (geof:geometryType(?literal) as ?gtype)
}
"""),["literal"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))


Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gtype,Polygon


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:getSRID Function

In [361]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?srid
WHERE {
  my:A geo:hasDefaultGeometry ?geom .
  ?geom geo:asWKT ?literal .
  BIND (geof:getSRID(?literal) as ?srid)
}
"""),["literal","envelope"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))


Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
srid,http://www.opengis.net/def/crs/OGC/1.3/CRS84


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:intersection Function

In [362]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?intersection
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:intersection(?aLiteral, ?dLiteral) as ?intersection)
}
"""),["aLiteral","dLiteral","intersection"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GEOMTOLIT: POLYGON ((-83.2 34.1, -83.3 34.1, -83.3 34.2, -83.2 34.2, -83.2 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
intersection,"POLYGON ((-83.2 34.1, -83.3 34.1, -83.3 34.2, -83.2 34.2, -83.2 34.1))"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:relate Function

In [363]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?relate) as ?relates) (xsd:boolean(?relate2) as ?relates2)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  # "T*****FF*" refers to a 'contains' relation in DE-9IM
  BIND (geof:relate(?aLiteral, ?bLiteral, "T*****FF*"^^xsd:string) as ?relate)
  BIND (geof:relate(?aLiteral, ?bLiteral, "F*****FF*"^^xsd:string) as ?relate2)
}
"""),["aLiteral","bLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
relates,true
relates2,false


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:symDifference Function

In [364]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?sdifference
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:symDifference(?aLiteral, ?dLiteral) as ?sdifference)
}
"""),["aLiteral","dLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GEOMTOLIT: MULTIPOLYGON (((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1)), ((-83.2 34.1, -83.2 34.2, -83.1 34.2, -83.1 34, -83.3 34, -83.3 34.1, -83.2 34.1)))
<class 'shapely.geometry.multipolygon.MultiPolygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
sdifference,"MULTIPOLYGON (((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1)), ((-83.2 34.1, -83.2 34.2, -83.1 34.2, -83.1 34, -83.3 34, -83.3 34.1, -83.2 34.1)))"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

### geof:union Function

In [365]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?union
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:union(?aLiteral, ?dLiteral) as ?union)
}
"""),["aLiteral","dLiteral"],Map(center=mapcenter, zoom=4))

LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
GString: Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
LString: <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GString: Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))
GEOMTOLIT: POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.1 34.2, -83.1 34, -83.3 34, -83.3 34.1, -83.6 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
union,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.1 34.2, -83.1 34, -83.3 34, -83.3 34.1, -83.6 34.1))"


Map(center=[24, 115], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…